# Politicians Network — Data Collection

Builds a **popularity-ranked** network of modern politicians (relevant after 1980).


In [1]:
import requests
import time
import json
import re
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from tqdm import tqdm
import threading
import random


## 1 — Shared session & constants

In [2]:
S = requests.Session()
S.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

BASE_WIKI  = "https://en.wikipedia.org/w/api.php"
SPARQL_URL = "https://query.wikidata.org/sparql"

SKIP_PREFIXES = ("Special:", "Main_Page", "Wikipedia:", "Help:", "File:", "Portal:")


def get_with_retry(url, params=None, session=S, max_retries=6):
    """GET with exponential backoff on 429 / 5xx."""
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=30)
            if r.status_code == 429:
                wait = 2 ** attempt + random.uniform(0.5, 1.5)
                print(f"  Rate-limited, waiting {wait:.1f}s…")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise RuntimeError(f"Failed after {max_retries} retries: {url}") from e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


## 2 — SPARQL executor & helpers

In [3]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "WikidataResearchBot/1.0 (kerem.ozemre@icloud.com)"})


def run_sparql(query, retries=5, base_sleep=1.5):
    for attempt in range(retries):
        try:
            r = SESSION.post(
                SPARQL_URL,
                data={"query": query},
                headers={"Accept": "application/json",
                         "User-Agent": "WikidataResearchBot/1.0"},
                timeout=60,
            )
            if r.status_code == 429:
                time.sleep(base_sleep * (2 ** attempt))
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f"SPARQL failed: {e}")
            time.sleep(base_sleep * (2 ** attempt))
    raise RuntimeError("SPARQL query failed after retries")


def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def parse_wikidata_date(raw):
    """Extract YYYY-MM-DD from a Wikidata timestamp string."""
    if not raw:
        return None
    m = re.match(r"[+-]?(\d{4}-\d{2}-\d{2})", raw)
    return m.group(1) if m else None


def extract_qid(uri):
    return uri.split("/")[-1]


## 3 — Popularity-ranked seed fetcher

Queries Wikidata for politicians ordered by **sitelink count** — the number of Wikipedia
language editions that cover them. This is the best available proxy for global notability
without hitting the pageviews API.

Filters: human (`P31=Q5`), politician (`P106=Q82955`), born ≤ 1970,
alive or died ≥ 1980.  
Remove `wdt:P27 wd:Q30` to go global instead of US-only.


In [4]:
def make_seed_query(limit=500, offset=0):
    return f"""
    SELECT DISTINCT ?person (COUNT(?sitelink) AS ?links) WHERE {{
      ?person wdt:P31 wd:Q5 ;       # person
              wdt:P106 wd:Q82955 . # occupation: politician

      ?person p:P39 ?stmt .         # held position (P39) statement
      ?stmt ps:P39 ?office .        
      ?office wdt:P17 wd:Q30 .          # office belongs to the United States

      OPTIONAL {{ ?person wdt:P569 ?birth. }} # date of birth
      OPTIONAL {{ ?person wdt:P570 ?death. }} # date of death  

      FILTER(
        BOUND(?birth) && YEAR(?birth) <= 1990
        && ( !BOUND(?death) || YEAR(?death) >= 1980 )
      )

      ?sitelink schema:about ?person .
    }}
    GROUP BY ?person
    ORDER BY DESC(?links) # most linked = most cross-wiki-covered
    LIMIT {limit}
    OFFSET {offset}
    """


def get_popular_politician_ids(max_seeds=800, page_size=500):
    """
    Return QID URI list for the most cross-wiki-covered politicians,
    paginating until max_seeds is reached or results are exhausted.
    page_size kept at 500 — GROUP BY + ORDER BY queries time out above ~500.
    """
    ids    = []
    offset = 0

    while len(ids) < max_seeds:
        query    = make_seed_query(limit=page_size, offset=offset)
        data     = run_sparql(query)
        bindings = data["results"]["bindings"]
        if not bindings:
            break
        batch = [b["person"]["value"] for b in bindings]
        ids.extend(batch)
        print(f"  Collected {len(ids)} seeds…")
        offset += page_size
        time.sleep(2)   # GROUP BY queries are heavier — be polite

    return ids[:max_seeds]


## 4 — Label fetcher

Resolves QIDs → English Wikipedia article titles.
The link-walker needs these titles to look up pages.
Also used to refresh titles for neighbor nodes discovered in expansion passes.


In [5]:
def fetch_labels(qids, batch_size=400):
    """
    Fetch English Wikipedia titles for a list of QIDs.
    Returns {wikipedia_title: qid} for QIDs that have an enwiki article.
    """
    name_to_qid = {}

    for chunk in tqdm(list(chunked(qids, batch_size)), desc="Fetching labels"):
        values = " ".join(f"wd:{q}" for q in chunk)
        query  = f"""
        SELECT ?person ?article WHERE {{
          VALUES ?person {{ {values} }}
          ?article schema:about ?person ;
                   schema:isPartOf <https://en.wikipedia.org/> .
        }}
        """
        data = run_sparql(query)
        for b in data["results"]["bindings"]:
            qid   = b["person"]["value"].split("/")[-1]
            title = (b["article"]["value"]
                     .replace("https://en.wikipedia.org/wiki/", "")
                     .replace("_", " "))
            name_to_qid[title] = qid
        time.sleep(1)

    print(f"  {len(name_to_qid)} QIDs have an English Wikipedia article")
    return name_to_qid


## 5 — Wikipedia link-walker (degree computation)

In [6]:
# ── Rate-limit governor ────────────────────────────────────────────────────────
_WIKI_SEM      = threading.Semaphore(4)
_BACKOFF_LOCK  = threading.Lock()
_BACKOFF_UNTIL = 0.0

def _wait_for_backoff():
    with _BACKOFF_LOCK:
        target = _BACKOFF_UNTIL
    remaining = target - time.monotonic()
    if remaining > 0:
        time.sleep(remaining)

def _set_backoff(seconds):
    global _BACKOFF_UNTIL
    with _BACKOFF_LOCK:
        _BACKOFF_UNTIL = max(_BACKOFF_UNTIL, time.monotonic() + seconds)


# ── Fetch all outgoing links from a Wikipedia page ────────────────────────────
def get_page_links(title):
    session = requests.Session()
    session.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

    links  = []
    params = {
        "action": "query", "titles": title, "prop": "links",
        "pllimit": "500", "plnamespace": "0", "format": "json",
    }
    while True:
        r    = get_with_retry(BASE_WIKI, params=params, session=session)
        data = r.json()
        for page in data.get("query", {}).get("pages", {}).values():
            for lk in page.get("links", []):
                links.append(lk["title"])
        cont = data.get("continue", {})
        if not cont:
            break
        params["plcontinue"] = cont["plcontinue"]
        time.sleep(0.2)
    return links


# ── Thread-safe caches ─────────────────────────────────────────────────────────
_title_lock = threading.Lock()
_qid_lock   = threading.Lock()

def resolve_titles_to_qids_cached(titles, cache, batch_size=50):
    with _title_lock:
        uncached = [t for t in titles if t not in cache]
    for i in range(0, len(uncached), batch_size):
        batch  = uncached[i:i + batch_size]
        params = {
            "action": "query", "titles": "|".join(batch),
            "prop": "pageprops", "ppprop": "wikibase_item", "format": "json",
        }
        r     = get_with_retry(BASE_WIKI, params=params)
        pages = r.json()["query"]["pages"]
        norm  = {n["from"]: n["to"] for n in r.json()["query"].get("normalized", [])}
        result = {}
        for page in pages.values():
            qid    = page.get("pageprops", {}).get("wikibase_item")
            ptitle = page.get("title", "")
            result[ptitle] = qid
        for orig, norm_title in norm.items():
            if norm_title in result:
                result[orig] = result[norm_title]
        with _title_lock:
            cache.update(result)
        time.sleep(0.1)
    with _title_lock:
        return {t: cache.get(t) for t in titles}


def filter_politician_qids_cached(qids, cache, batch_size=200):
    with _qid_lock:
        unchecked = [q for q in qids if q not in cache["seen"]]
    for i in range(0, len(unchecked), batch_size):
        batch  = unchecked[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query  = f"""
        SELECT ?person WHERE {{
          VALUES ?person {{ {values} }}
          ?person wdt:P106 wd:Q82955 .

          # Must have held a US office
          ?person p:P39 ?stmt .
          ?stmt ps:P39 ?office .
          ?office wdt:P17 wd:Q30 .

          # Same date filter as seeds
          OPTIONAL {{ ?person wdt:P569 ?birth. }}
          OPTIONAL {{ ?person wdt:P570 ?death. }}
          FILTER(
            BOUND(?birth) && YEAR(?birth) <= 1990
            && ( !BOUND(?death) || YEAR(?death) >= 1980 )
          )
        }}
        """
        data  = run_sparql(query)
        found = [item["person"]["value"].split("/")[-1]
                 for item in data["results"]["bindings"]]
        with _qid_lock:
            cache["politicians"].update(found)
            cache["seen"].update(batch)
        time.sleep(0.5)
    with _qid_lock:
        return cache["politicians"]


# ── Serial walker with adaptive throttle ──────────────────────────────────────
def compute_politician_degrees(name_to_qid, title_qid_cache=None, politician_cache=None):
    """
    Walk Wikipedia pages for each politician in name_to_qid.
    Returns (degrees, neighbours) dicts keyed by QID.
    Caches are shared across calls so repeated QIDs are only looked up once.
    """
    if title_qid_cache is None:
        title_qid_cache = {}
    if politician_cache is None:
        politician_cache = {"seen": set(), "politicians": set()}

    degrees   = {}
    neighbors = {}
    delay     = 0.3

    for idx, (title, qid) in enumerate(tqdm(name_to_qid.items(), desc="Walking pages"), 1):
        linked_titles = []
        for attempt in range(6):
            try:
                linked_titles = get_page_links(title)
                delay = max(0.1, delay * 0.95)
                break
            except Exception as e:
                if "429" in str(e):
                    delay = min(30, delay * 2)
                    print(f"  429 on '{title}' — slowing to {delay:.1f}s")
                    time.sleep(delay)
                elif attempt == 5:
                    print(f"  Giving up on '{title}': {e}")
                    break
                else:
                    time.sleep(2 ** attempt)

        if not linked_titles:
            degrees[qid]   = 0
            neighbors[qid] = []
        else:
            t2q            = resolve_titles_to_qids_cached(linked_titles, title_qid_cache)
            linked_qids    = [q for q in t2q.values() if q]
            pol_set        = filter_politician_qids_cached(linked_qids, politician_cache) if linked_qids else set()
            pol_qids       = [q for q in linked_qids if q in pol_set]
            degrees[qid]   = len(pol_qids)
            neighbors[qid] = pol_qids

        time.sleep(delay)

        if idx % 50 == 0:
            print(f"  [{idx}/{len(name_to_qid)}] delay={delay:.2f}s | "
                  f"title cache={len(title_qid_cache)} | "
                  f"qid cache={len(politician_cache['seen'])}")

    return degrees, neighbors


## 6 — Feature fetcher

In [10]:
def make_details_query(person_ids):
    values = " ".join(f"wd:{p.split('/')[-1]}" for p in person_ids)
    return f"""
    SELECT
      ?person ?personLabel ?positionLabel
      ?start ?end ?natLabel ?birth ?death
      ?partyLabel ?genderLabel ?educationLabel ?stateLabel ?article
    WHERE {{
      VALUES ?person {{ {values} }}

      ?person p:P39 ?statement .
      ?statement ps:P39 ?position .
      OPTIONAL {{ ?statement pq:P580 ?start. }}
      OPTIONAL {{ ?statement pq:P582 ?end. }}

      OPTIONAL {{ ?person wdt:P27 ?nat. }}
      OPTIONAL {{ ?person wdt:P569 ?birth. }}
      OPTIONAL {{ ?person wdt:P570 ?death. }}
      OPTIONAL {{ ?person wdt:P102 ?party. }}
      OPTIONAL {{ ?person wdt:P21 ?gender. }}
      OPTIONAL {{ ?person wdt:P69 ?education. }}
      OPTIONAL {{ ?position wdt:P131 ?state. }}
      OPTIONAL {{
        ?article schema:about ?person ;
                 schema:isPartOf <https://en.wikipedia.org/> .
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """


def fetch_details(person_ids, batch_size=50):
    rows = []
    for i, chunk in enumerate(chunked(person_ids, batch_size), 1):
        print(f"  Details batch {i}/{-(-len(person_ids)//batch_size)} ({len(chunk)} people)")
        data = run_sparql(make_details_query(chunk))
        for item in data["results"]["bindings"]:
            rows.append({
                "wikidata":      extract_qid(item["person"]["value"]),
                "name":          item.get("personLabel",   {}).get("value"),
                "position":      item.get("positionLabel", {}).get("value"),
                "start":         parse_wikidata_date(item.get("start",  {}).get("value")),
                "end":           parse_wikidata_date(item.get("end",    {}).get("value")),
                "nationality":   item.get("natLabel",      {}).get("value"),
                "birth_date":    parse_wikidata_date(item.get("birth",  {}).get("value")),
                "death_date":    parse_wikidata_date(item.get("death",  {}).get("value")),
                "party":         item.get("partyLabel",    {}).get("value"),
                "gender":        item.get("genderLabel",   {}).get("value"),
                "education":     item.get("educationLabel",{}).get("value"),
                "state":         item.get("stateLabel",    {}).get("value"),
                "wikipedia_url": item.get("article",       {}).get("value"),
            })
        time.sleep(1.5)
    return rows


## 7 — Dedup & party helpers

In [11]:
POSITION_RANK = {
    "President of the United States": 1,
    "Vice President of the United States": 2,
    "Secretary of State": 3,
    "Secretary of Defense": 4,
    "Attorney General of the United States": 5,
    "Secretary of the Treasury": 6,
    "White House Chief of Staff": 7,
    "United States Senator": 8,
    "Speaker of the House of Representatives": 9,
    "House Majority Leader": 10,
    "Senate Majority Leader": 10,
    "United States Representative": 11,
    "Governor": 12,
    "Ambassador": 13,
    "Lieutenant Governor": 14,
    "State Senator": 15,
    "State Representative": 16,
    "Mayor": 17,
}

def position_priority(pos):
    if pos is None:
        return 999
    if pos in POSITION_RANK:
        return POSITION_RANK[pos]
    for key, rank in POSITION_RANK.items():
        if key.lower() in pos.lower():
            return rank
    return 998


def simplify_party(p):
    if p is None:
        return None

    p_lower = p.lower()

    # Major parties
    if "democratic" in p_lower:
        return "Democrat"
    if "republican" in p_lower:
        return "Republican"

    # Third / moderate parties
    if "independent" in p_lower:
        return "Independent"
    if "libertarian" in p_lower:
        return "Libertarian"
    if "green" in p_lower:
        return "Green"
    if "reform" in p_lower:
        return "Reform"
    if "constitution" in p_lower:
        return "Constitution"
    if "progressive" in p_lower:
        return "Progressive"
    if "socialist" in p_lower:
        return "Socialist"
    if "whig" in p_lower:
        return "Whig"
    if "federalist" in p_lower:
        return "Federalist"
    if "know nothing" in p_lower or "american party" in p_lower:
        return "Know Nothing"

    return "Other"


def dedup_df(df):
    """
    Per unique wikidata QID:
      - position  → most prestigious title (lowest POSITION_RANK score)
      - start     → earliest start date across all positions
      - end       → latest end date across all positions
      - all other columns (party, gender, education, etc.) → from the most prestigious row
    """
    df = df.copy()
    df["position_rank"] = df["position"].apply(position_priority)

    # ── Most prestigious row per person (for title + all metadata) ─────────────
    best = (df.sort_values("position_rank")
              .drop_duplicates(subset=["wikidata"], keep="first")
              .set_index("wikidata"))

    # ── Earliest start date per person ────────────────────────────────────────
    earliest_start = (df[df["start"].notna()]
                      .sort_values("start")
                      .drop_duplicates(subset=["wikidata"], keep="first")
                      .set_index("wikidata")["start"])

    # ── Latest end date per person ─────────────────────────────────────────────
    # Treat NaN end as still in office — represented as None, not overwritten
    latest_end = (df[df["end"].notna()]
                  .sort_values("end", ascending=False)
                  .drop_duplicates(subset=["wikidata"], keep="first")
                  .set_index("wikidata")["end"])

    # ── Assemble ───────────────────────────────────────────────────────────────
    best["career_start"] = best.index.map(earliest_start)
    best["career_end"]   = best.index.map(latest_end)

    # If someone has no end date at all, career_end stays None (still in office)
    # If someone has no start date at all, career_start stays None

    result = best.drop(columns=["start", "end", "position_rank"]).reset_index()
    result["party_simple"] = result["party"].apply(simplify_party)

    print(f"  {len(result)} unique politicians after dedup")
    print(f"  career_start filled: {result['career_start'].notna().sum()}")
    print(f"    career_end filled: {result['career_end'].notna().sum()} "
          f"({result['career_end'].isna().sum()} still in office or unknown)")
    return result

## 8 — End-to-end pipeline

```
run_pipeline(max_seeds=800, expand_neighbors=True)
```

| `expand_neighbors` | Expected nodes |
|---|---|
| `False` (pass 1 only) | ~2,500–3,500 |
| `True`  (pass 1 + 2)  | ~6,000–10,000 |

Shared caches are passed between expansion passes so no QID is resolved twice.
`fetch_details` runs **once** at the very end on the full combined node set.


In [12]:
def run_pipeline(max_seeds=800, expand_neighbors=False):
    # ── Phase 1: popularity-ranked seeds ──────────────────────────────────────
    print("=" * 60)
    print("Phase 1 — Fetching popularity-ranked seed IDs…")
    seed_uris = get_popular_politician_ids(max_seeds=max_seeds)
    seed_qids = {extract_qid(u) for u in seed_uris}
    print(f"  Unique seeds: {len(seed_qids)}")

    print("\nFetching Wikipedia titles for seeds…")
    seed_titles = fetch_labels(list(seed_qids))
    print(f"  Seeds with enwiki article: {len(seed_titles)}")

    # Shared caches — reused across both expansion passes
    title_qid_cache  = {}
    politician_cache = {"seen": set(), "politicians": set()}

    # ── Phase 2: link-walk seeds ───────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Phase 2 — Link-walking seed pages…")
    degrees, neighbours = compute_politician_degrees(
        seed_titles, title_qid_cache, politician_cache
    )

    neighbor_qids_1 = {q for conns in neighbours.values() for q in conns} - seed_qids
    all_qids        = seed_qids | neighbor_qids_1
    print(f"  Seeds              : {len(seed_qids)}")
    print(f"  New from pass 1    : {len(neighbor_qids_1)}")
    print(f"  Total after pass 1 : {len(all_qids)}")

    # ── Phase 3: optional second expansion pass ────────────────────────────────
    if expand_neighbors and neighbor_qids_1:
        print("\n" + "=" * 60)
        print("Phase 3 — Link-walking neighbor pages…")
        neighbor_titles = fetch_labels(list(neighbor_qids_1))
        degrees_2, neighbours_2 = compute_politician_degrees(
            neighbor_titles, title_qid_cache, politician_cache
        )

        neighbor_qids_2 = {q for conns in neighbours_2.values() for q in conns} - all_qids
        all_qids       |= neighbor_qids_2
        print(f"  New from pass 2    : {len(neighbor_qids_2)}")
        print(f"  Total after pass 2 : {len(all_qids)}")

        degrees.update(degrees_2)
        neighbours.update(neighbours_2)

    # ── Link-walk the non-seed neighbors too ──────────────────────────────────
    # Seeds already have degrees/neighbours from the walk above.
    # Non-seed nodes were only *discovered* as neighbors — we haven't walked
    # their pages yet, so we do that now to get their connections too.
    non_seed_qids   = all_qids - seed_qids
    non_seed_titles = fetch_labels(list(non_seed_qids))
    print(f"\nLink-walking {len(non_seed_titles)} non-seed pages for their connections…")
    degrees_ns, neighbours_ns = compute_politician_degrees(
        non_seed_titles, title_qid_cache, politician_cache
    )
    # Only add — don't overwrite seeds that were already walked
    for qid, deg in degrees_ns.items():
        if qid not in degrees:
            degrees[qid]   = deg
            neighbours[qid] = neighbours_ns[qid]

    # ── Phase 4: fetch features for everyone ──────────────────────────────────
    print("\n" + "=" * 60)
    print(f"Phase 4 — Fetching features for all {len(all_qids)} politicians…")
    all_uris = [f"http://www.wikidata.org/entity/{q}" for q in all_qids]
    rows     = fetch_details(all_uris)

    df = pd.DataFrame(rows)
    df = dedup_df(df)
    df["degree"]      = df["wikidata"].map(degrees)
    df["connections"] = df["wikidata"].map(neighbours).apply(
        lambda x: x if isinstance(x, list) else []
    )
    df["is_seed"] = df["wikidata"].isin(seed_qids)

    # ── Save edges to JSON ─────────────────────────────────────────────────────
    import json
    node_set     = set(df["wikidata"])
    clean_edges  = {
        qid: [n for n in conns if n in node_set]
        for qid, conns in neighbours.items()
        if qid in node_set
    }
    with open("politician_edges.json", "w") as f:
        json.dump(clean_edges, f)
    print(f"Saved → politician_edges.json  "
          f"({len(clean_edges)} nodes, "
          f"{sum(len(v) for v in clean_edges.values())} total edge entries)")

    # ── Save node CSV ──────────────────────────────────────────────────────────
    df.to_csv("politicians_network_nodes.csv", index=False)
    print(f"Saved → politicians_network_nodes.csv  ({len(df)} rows)")

    n_seeds = df["is_seed"].sum()
    print(f"\n{'=' * 60}")
    print(f"Pipeline complete.")
    print(f"  Seed nodes     : {n_seeds}")
    print(f"  Expanded nodes : {len(df) - n_seeds}")
    print(f"  Total nodes    : {len(df)}")
    return df


df = run_pipeline(max_seeds=500, expand_neighbors=False)


Phase 1 — Fetching popularity-ranked seed IDs…


RuntimeError: SPARQL failed: 502 Server Error: Bad Gateway for url: https://query.wikidata.org/sparql

## 9 — Save

In [ ]:
if not df.empty:
    print(f"Shape: {df.shape}")
    print(df.head(20).to_string())
else:
    print("Nothing to save — DataFrame is empty.")


Shape: (5901, 17)
     wikidata                   name                                                   position    nationality  birth_date  death_date             party  gender                                        education                                      state                                        wikipedia_url career_start  career_end party_simple  degree                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
df["party"] = df["party"].apply(simplify_party)
df

,wikidata,name,position,nationality,birth_date,death_date,party,gender,education,state,wikipedia_url,career_start,career_end,party_simple,degree,connections,is_seed
0,Q76,Barack Obama,President of the United States,United States,1961-08-04,None,Democrat,male,Centaurus High School,Hawaii,https://en.wikipedia.org/wiki/Barack_Obama,1997-01-08,2017-01-20,Democrat,213,"[Q55603085, Q122841, Q319084, Q19673, Q1886056...",True
1,Q42770663,Keisha Lance Bottoms,Senior Advisor to the President of the United ...,United States,1970-01-18,None,Democrat,female,Georgia State University College of Law,Georgia,https://en.wikipedia.org/wiki/Keisha_Lance_Bot...,2018-01-02,2023-04-01,Democrat,25,"[Q292340, Q109845767, Q959635, Q4908401, Q4964...",False
2,Q1290476,John Podesta,Senior Advisor to the President of the United ...,United States,1949-01-08,None,Democrat,male,Lane Technical College Prep High School,United States,https://en.wikipedia.org/wiki/John_Podesta,1997-01-20,2025-01-20,Democrat,113,"[Q19673, Q107957, Q292340, Q504029, Q11673, Q2...",False
3,Q317521,Elon Musk,Senior Advisor to the President of the United ...,United States,1971-06-28,None,Other,male,Bryanston High School,Pretoria,https://en.wikipedia.org/wiki/Elon_Musk,2002-01-01,2025-05-30,Other,29,"[Q76, Q461657, Q201795, Q213122, Q1124, Q45744...",True
4,Q9582,Gerald Ford,President of the United States,United States,1913-07-14,2006-12-26,Republican,male,Yale Law School,United States,https://en.wikipedia.org/wiki/Gerald_Ford,1949-01-03,1977-01-20,Republican,287,"[Q329910, Q350663, Q122841, Q19673, Q888927, Q...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5896,Q16846619,"R. Clayton Mitchell, Jr.",member of the Maryland House of Delegates,United States,1936-04-16,2019-06-14,Democrat,male,None,Chestertown,https://en.wikipedia.org/wiki/R._Clayton_Mitch...,NaN,NaN,Democrat,2,"[Q5049091, Q7174061]",False
5897,Q16195953,Richard Corcoran,member of the Florida House of Representatives,United States,1965-03-16,None,Republican,male,Saint Leo University,Toronto,https://en.wikipedia.org/wiki/Richard_Corcoran,NaN,NaN,Republican,46,"[Q350663, Q4730479, Q59660252, Q999912, Q16149...",False
5898,Q45311991,Lena King Lee,member of the Maryland House of Delegates,United States,1906-07-14,2006-08-24,Democrat,female,University of Maryland Francis King Carey Scho...,Sumter County,https://en.wikipedia.org/wiki/Lena_King_Lee,NaN,NaN,Democrat,31,"[Q4685671, Q4750503, Q4802163, Q4858694, Q2611...",False
5899,Q58974608,Wally Straughn,member of the Arizona House of Representatives,United States,1957-01-01,None,Democrat,male,None,United States,https://en.wikipedia.org/wiki/Wally_Straughn,NaN,NaN,Democrat,2,"[Q1175343, Q1556541]",False


In [ ]:
career_end_dt = pd.to_datetime(df["career_end"], errors='coerce')
mask = career_end_dt >= pd.Timestamp("1980-01-01")  # Keep entries on or after 1970

df = df[mask]
df

,wikidata,name,position,nationality,birth_date,death_date,party,gender,education,state,wikipedia_url,career_start,career_end,party_simple,degree,connections,is_seed
0,Q76,Barack Obama,President of the United States,United States,1961-08-04,None,Democrat,male,Centaurus High School,Hawaii,https://en.wikipedia.org/wiki/Barack_Obama,1997-01-08,2017-01-20,Democrat,213,"[Q55603085, Q122841, Q319084, Q19673, Q1886056...",True
1,Q42770663,Keisha Lance Bottoms,Senior Advisor to the President of the United ...,United States,1970-01-18,None,Democrat,female,Georgia State University College of Law,Georgia,https://en.wikipedia.org/wiki/Keisha_Lance_Bot...,2018-01-02,2023-04-01,Democrat,25,"[Q292340, Q109845767, Q959635, Q4908401, Q4964...",False
2,Q1290476,John Podesta,Senior Advisor to the President of the United ...,United States,1949-01-08,None,Democrat,male,Lane Technical College Prep High School,United States,https://en.wikipedia.org/wiki/John_Podesta,1997-01-20,2025-01-20,Democrat,113,"[Q19673, Q107957, Q292340, Q504029, Q11673, Q2...",False
3,Q317521,Elon Musk,Senior Advisor to the President of the United ...,United States,1971-06-28,None,Other,male,Bryanston High School,Pretoria,https://en.wikipedia.org/wiki/Elon_Musk,2002-01-01,2025-05-30,Other,29,"[Q76, Q461657, Q201795, Q213122, Q1124, Q45744...",True
5,Q6279,Joe Biden,President of the United States,United States,1942-11-20,None,Democrat,male,Syracuse University,Scranton,https://en.wikipedia.org/wiki/Joe_Biden,1970-11-04,2025-01-20,Democrat,304,"[Q55603085, Q106250968, Q319084, Q19673, Q2023...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5890,Q473061,Jim Ramstad,member of the United States House of Represent...,United States,1946-05-06,2020-11-05,Republican,male,George Washington University,United States,https://en.wikipedia.org/wiki/Jim_Ramstad,1981-01-06,2009-01-03,Republican,61,"[Q888927, Q580489, Q22237, Q489070, Q58324174,...",False
5891,Q350663,Adam Putnam,member of the United States House of Represent...,United States,1974-07-31,None,Republican,male,University of Florida,Florida,https://en.wikipedia.org/wiki/Adam_Putnam,1996-01-01,2019-01-08,Republican,159,"[Q4661833, Q4723052, Q1281084, Q202350, Q17586...",False
5892,Q733585,Tony Snow,White House Press Secretary,United States,1955-06-01,2008-07-12,Republican,male,Princeton High School,Berea,https://en.wikipedia.org/wiki/Tony_Snow,2006-05-10,2007-09-14,Republican,18,"[Q1263856, Q114322, Q434317, Q23505, Q1655924,...",False
5893,Q963419,Robin Beard,member of the United States House of Represent...,United States,1939-08-21,2007-06-16,Republican,male,Vanderbilt University,Knoxville,https://en.wikipedia.org/wiki/Robin_Beard,1973-01-03,1983-01-03,Republican,53,"[Q19673, Q577160, Q111623114, Q517219, Q809197...",False


In [ ]:
df.to_csv("politicians_network_nodes2.csv", index=False)

In [ ]:
import networkx as nx
import ast
def parse_connections(val):
    # If it's already a list, tuple, or set, convert to set directly
    if isinstance(val, (list, tuple, set)):
        return set(val)
    
    # Check for NaN (only safe if val is not a list)
    if pd.isna(val):
        return set()
    
    # If it's a string, try to parse it
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return set(parsed)
        except:
            # If literal_eval fails, maybe it's comma-separated
            return set([x.strip() for x in val.split(",") if x.strip()])
    
    return set()

id_to_connections = {
    k: parse_connections(v)
    for k, v in zip(df["wikidata"], df["connections"])
}

# ── Build graph ────────────────────────────────────────────────────────────────
G = nx.Graph()

# Node attributes: store every DataFrame feature on the node
FEATURE_COLS = [
    "wikidata", "name", "position", "nationality", "birth_date", "death_date", "party", "gender", "education", "state", "wikipedia_url", "career_start", "career_end", "party_simple", "degree", "connections", "is_seed"
]

# Add nodes first so isolated politicians are still included
for _, row in df.iterrows():
    qid   = row["wikidata"]
    attrs = {col: row.get(col) for col in FEATURE_COLS if col in df.columns}
    G.add_node(qid, **attrs)

# Add edges
for person, conns in id_to_connections.items():
    for other in conns:
        if other in id_to_connections:
            G.add_edge(person, other)

# ── Largest connected component ────────────────────────────────────────────────
largest_cc = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(largest_cc).copy()
print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
print(f"Largest connected component: {len(largest_cc)} politicians")

# ── Quick sanity-check: node attributes ───────────────────────────────────────
sample_qid = next(iter(largest_cc))
print(f"\nSample node ({sample_qid}) attributes:")
for k, v in G_lcc.nodes[sample_qid].items():
    print(f"  {k}: {v}")

Nodes: 3337 | Edges: 166594
Largest connected component: 3337 politicians

Sample node (Q675869) attributes:
  wikidata: Q675869
  name: Jim Costa
  position: member of the State Senate of California
  nationality: United States
  birth_date: 1952-04-13
  death_date: None
  party: Democrat
  gender: male
  education: San Joaquin Memorial High School
  state: Fresno
  wikipedia_url: https://en.wikipedia.org/wiki/Jim_Costa
  career_start: 1978-01-01
  career_end: 2021-01-03
  party_simple: Democrat
  degree: 621
  connections: ['Q4661833', 'Q3000097', 'Q350843', 'Q350916', 'Q134612342', 'Q373443', 'Q4685569', 'Q19673', 'Q749039', 'Q3740782', 'Q4717593', 'Q55223040', 'Q2648018', 'Q4733597', 'Q120699', 'Q18684027', 'Q3389105', 'Q138142530', 'Q39791512', 'Q495050', 'Q102277679', 'Q21257859', 'Q517649', 'Q3828645', 'Q16199304', 'Q506639', 'Q111623114', 'Q16196315', 'Q21062653', 'Q58324174', 'Q3917251', 'Q291193', 'Q112805534', 'Q572941', 'Q125935301', 'Q59179980', 'Q60713905', 'Q101196462', 